<a href="https://www.kaggle.com/code/ab0y04/skin-lesion-imagenet?scriptVersionId=342055831" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# ===== FULL REBUILD (new session) + STAGE 16 FOLD 1: RESNET50 + SAFE MERGE =====
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'   # MUST precede tensorflow import
import random, gc
import numpy as np
import pandas as pd
import tensorflow as tf

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print(f"Seed {SEED} set, TF {tf.__version__}, tf.keras module: {tf.keras.__name__}")
assert 'tf_keras' in tf.keras.__name__, "STOP: Keras 3 active, not legacy. Restart before continuing."

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input as res_pre
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, recall_score, confusion_matrix
print("Imports ready")

FINAL_CLASSES = ['bcc', 'bkl', 'df', 'melanoma', 'nevus', 'vasc']
NUM_CLASSES = 6
IMG_SIZE, BATCH_SIZE = 224, 32
N_FOLDS, CURRENT_FOLD = 5, 1
AUG = dict(rotation_range=20, width_shift_range=0.1, height_shift_range=0.1,
           horizontal_flip=True, zoom_range=0.1)
print("Config loaded")

CV_ASSIGN_PATH = '/kaggle/input/datasets/ab0y04/cvfoldassignments/cv_fold_assignments.csv'
assert os.path.exists(CV_ASSIGN_PATH), f"STOP: file not found at {CV_ASSIGN_PATH}, check the dataset is attached"
cv_assignments = pd.read_csv(CV_ASSIGN_PATH)
print(f"Loaded cv_fold_assignments.csv: {len(cv_assignments):,} rows (expect 23,836)")
assert len(cv_assignments) == 23836, "STOP: row count mismatch"

def build_fold_split(cv_assignments, fold_num, seed=42):
    test_df = cv_assignments[cv_assignments['fold'] == fold_num].reset_index(drop=True)
    remaining = cv_assignments[cv_assignments['fold'] != fold_num].reset_index(drop=True)
    remaining = remaining.copy()
    fallback = pd.Series('unlinked_' + remaining.index.astype(str), index=remaining.index)
    remaining['_split_key'] = remaining['group_id'].fillna(fallback)
    groups = remaining.groupby('_split_key')['label'].first().reset_index()
    tr_groups, va_groups = train_test_split(groups, test_size=0.15, stratify=groups['label'], random_state=seed)
    train_df = remaining[remaining['_split_key'].isin(tr_groups['_split_key'])].drop(columns=['_split_key']).reset_index(drop=True)
    val_df = remaining[remaining['_split_key'].isin(va_groups['_split_key'])].drop(columns=['_split_key']).reset_index(drop=True)
    return train_df, val_df, test_df

train_df, val_df, test_df = build_fold_split(cv_assignments, CURRENT_FOLD, seed=SEED)
print(f"\n===== FOLD {CURRENT_FOLD} SPLIT =====")
print(f"Train {len(train_df):,} | Val {len(val_df):,} | Test {len(test_df):,}")

test_groups = set(test_df['group_id'].dropna())
train_groups = set(train_df['group_id'].dropna())
val_groups = set(val_df['group_id'].dropna())
assert test_groups.isdisjoint(train_groups) and test_groups.isdisjoint(val_groups) and train_groups.isdisjoint(val_groups), \
    "STOP: FOLD 1 LEAKAGE detected"
print("Leakage check: PASS")

cls = np.array(FINAL_CLASSES)
cw = compute_class_weight('balanced', classes=cls, y=train_df['label'])
w_map = {c: w for c, w in zip(FINAL_CLASSES, cw)}
train_df['sample_weight'] = train_df['label'].map(w_map)
print(f"Fold {CURRENT_FOLD} class_weight:", {c: round(w,3) for c,w in zip(cls, cw)})
print("(expect: bcc 1.184, df 16.284, nevus 0.309, matching every prior fold-1 run)")

def make_fold_gens(preprocess_fn):
    train_idg = ImageDataGenerator(preprocessing_function=preprocess_fn, **AUG)
    eval_idg  = ImageDataGenerator(preprocessing_function=preprocess_fn)
    common = dict(x_col='image_path', y_col='label', target_size=(IMG_SIZE,IMG_SIZE),
                  batch_size=BATCH_SIZE, class_mode='categorical', classes=FINAL_CLASSES)
    tr = train_idg.flow_from_dataframe(train_df, shuffle=True, seed=SEED, weight_col='sample_weight', **common)
    va = eval_idg.flow_from_dataframe(val_df, shuffle=False, **common)
    te = eval_idg.flow_from_dataframe(test_df, shuffle=False, **common)
    return tr, va, te

def build_pretrained(base_class, num_classes=6, shape=(224,224,3)):
    base = base_class(include_top=False, weights='imagenet', input_shape=shape)
    model = Sequential([base, GlobalAveragePooling2D(), Dense(256,activation='relu'),
                          Dropout(0.3), Dense(num_classes,activation='softmax')])
    return model, base

def macro_specificity(y_true, y_pred, n_classes):
    cm = confusion_matrix(y_true, y_pred, labels=range(n_classes))
    total = cm.sum(); specs = []
    for i in range(n_classes):
        tp = cm[i,i]; fn = cm[i,:].sum()-tp; fp = cm[:,i].sum()-tp
        tn = total-tp-fn-fp
        specs.append(tn/(tn+fp) if (tn+fp)>0 else np.nan)
    return np.nanmean(specs)

print(f"\n===== Fold {CURRENT_FOLD} setup verified. Training ResNet50 only. =====\n")

fold_results = []

# ---------- RESNET50 ----------
try:
    print(f"{'='*60}\nFOLD {CURRENT_FOLD}: ResNet50\n{'='*60}")
    tr, va, te = make_fold_gens(res_pre)
    print("class_indices:", te.class_indices)

    model, base = build_pretrained(ResNet50)
    log_file = f'/kaggle/working/cv_f{CURRENT_FOLD}_res_log.csv'

    base.trainable = False
    model.compile(Adam(1e-3), 'categorical_crossentropy', ['accuracy'])
    model.fit(tr, validation_data=va, epochs=10, callbacks=[CSVLogger(log_file, append=False)], verbose=1)
    print("Phase 1 sanity:", model.evaluate(te, verbose=0))

    base.trainable = True
    model.compile(Adam(1e-5), 'categorical_crossentropy', ['accuracy'])
    cbs = [EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True),
           ModelCheckpoint(f'/kaggle/working/cv_f{CURRENT_FOLD}_res.keras', monitor='val_accuracy', save_best_only=True),
           CSVLogger(log_file, append=True)]
    model.fit(tr, validation_data=va, epochs=60, callbacks=cbs, verbose=1)

    y_true = np.asarray(te.classes)
    y_prob = model.predict(te, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)
    np.savez(f'/kaggle/working/cv_f{CURRENT_FOLD}_preds_res.npz', y_true=y_true, y_pred=y_pred, y_prob=y_prob)

    reloaded = load_model(f'/kaggle/working/cv_f{CURRENT_FOLD}_res.keras')
    verify_acc = reloaded.evaluate(te, verbose=0)[1]
    live_acc = accuracy_score(y_true, y_pred)
    print(f"Checkpoint verify: reloaded acc {verify_acc:.4f} vs live acc {live_acc:.4f}  match={abs(verify_acc-live_acc)<1e-3}")
    del reloaded

    fold_results.append(dict(fold=CURRENT_FOLD, arch='res', accuracy=live_acc,
        macro_f1=f1_score(y_true,y_pred,average='macro'),
        macro_auc=roc_auc_score(np.eye(NUM_CLASSES)[y_true], y_prob, average='macro', multi_class='ovr'),
        macro_sensitivity=recall_score(y_true,y_pred,average='macro'),
        macro_specificity=macro_specificity(y_true,y_pred,NUM_CLASSES), n_test=len(y_true)))
    print(f"RESULT: {fold_results[-1]}")
    del model, base; gc.collect(); tf.keras.backend.clear_session()
except Exception as e:
    print(f"!!! FOLD {CURRENT_FOLD} ResNet50 FAILED: {type(e).__name__}: {e}")
    import traceback; traceback.print_exc()
    gc.collect(); tf.keras.backend.clear_session()

# ---------- SAFE MERGE: check before writing, use the confirmed real path, no blind overwrite ----------
new_df = pd.DataFrame(fold_results)
PRIOR_RESULTS_PATH = '/kaggle/input/datasets/ab0y04/k-foold-validation-fold-1/cv_f1_results.csv'

print(f"\n{'='*60}\nMERGE CHECK\n{'='*60}")
print("Current /kaggle/working contents:", os.listdir('/kaggle/working'))
print(f"Prior results file exists at expected path: {os.path.exists(PRIOR_RESULTS_PATH)}")

if os.path.exists(PRIOR_RESULTS_PATH):
    prior_df = pd.read_csv(PRIOR_RESULTS_PATH)
    print(f"\nPrior results ({len(prior_df)} rows):")
    print(prior_df.to_string(index=False))
    combined = pd.concat([prior_df, new_df], ignore_index=True).drop_duplicates(subset=['fold','arch'], keep='last')
else:
    print("\nWARNING: prior results file not found at the expected path. Saving ResNet50's result alone.")
    print("You will need to manually merge with custom/eff/mob using this session's output plus the")
    print("prior conversation record if the merge needs to happen elsewhere.")
    combined = new_df

combined.to_csv('/kaggle/working/cv_f1_results.csv', index=False)
print(f"\n{'='*60}\nFOLD {CURRENT_FOLD} FINAL STATUS\n{'='*60}")
print(combined.to_string(index=False))
print(f"\nRows: {len(combined)}/4 architectures")
print("Saved: /kaggle/working/cv_f1_results.csv")

2026-08-13 03:28:12.469055: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1786591692.651245      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1786591692.706337      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1786591693.149354      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1786591693.149393      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1786591693.149396      23 computation_placer.cc:177] computation placer alr

Seed 42 set, TF 2.19.0, tf.keras module: tf_keras.api._v2.keras
Imports ready
Config loaded
Loaded cv_fold_assignments.csv: 23,836 rows (expect 23,836)

===== FOLD 1 SPLIT =====
Train 16,121 | Val 2,882 | Test 4,833
Leakage check: PASS
Fold 1 class_weight: {np.str_('bcc'): np.float64(1.184), np.str_('bkl'): np.float64(1.514), np.str_('df'): np.float64(16.284), np.str_('melanoma'): np.float64(0.88), np.str_('nevus'): np.float64(0.309), np.str_('vasc'): np.float64(15.442)}
(expect: bcc 1.184, df 16.284, nevus 0.309, matching every prior fold-1 run)

===== Fold 1 setup verified. Training ResNet50 only. =====

FOLD 1: ResNet50
Found 16121 validated image filenames belonging to 6 classes.
Found 2882 validated image filenames belonging to 6 classes.
Found 4833 validated image filenames belonging to 6 classes.
class_indices: {'bcc': 0, 'bkl': 1, 'df': 2, 'melanoma': 3, 'nevus': 4, 'vasc': 5}


I0000 00:00:1786591766.984413      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1786591766.990484      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


94765736/94765736 [==============================] - 1s 0us/step
Epoch 1/10


I0000 00:00:1786591775.170211      67 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1786591777.413908      66 service.cc:152] XLA service 0x7a2ac87dc150 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1786591777.413941      66 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1786591777.413945      66 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1786591777.665037      66 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


504/504 [==============================] - 633s 1s/step - loss: 1.4932 - accuracy: 0.4663 - val_loss: 0.9035 - val_accuracy: 0.6645
Epoch 2/10
504/504 [==============================] - 411s 816ms/step - loss: 1.1596 - accuracy: 0.5369 - val_loss: 1.1239 - val_accuracy: 0.5350
Epoch 3/10
504/504 [==============================] - 397s 789ms/step - loss: 1.0478 - accuracy: 0.5505 - val_loss: 0.9651 - val_accuracy: 0.6367
Epoch 4/10
504/504 [==============================] - 400s 793ms/step - loss: 1.0151 - accuracy: 0.5582 - val_loss: 1.2504 - val_accuracy: 0.4986
Epoch 5/10
504/504 [==============================] - 401s 797ms/step - loss: 0.9438 - accuracy: 0.5801 - val_loss: 1.0912 - val_accuracy: 0.5604
Epoch 6/10
504/504 [==============================] - 399s 791ms/step - loss: 0.9207 - accuracy: 0.5893 - val_loss: 1.1010 - val_accuracy: 0.5704
Epoch 7/10
504/504 [==============================] - 401s 795ms/step - loss: 0.8898 - accuracy: 0.5967 - val_loss: 0.9713 - val_accuracy: